Example trapped resonance objective function computation and optimization script

In [1]:
# Objective function computation

import desc.io
from desc.objectives import (
    TrappedResonance
)
import numpy as np
from desc.examples import get


# eq = desc.io.load("equil_Helios_E0956_R80_B60_DESC_fixed.h5")
eq = get('HELIOTRON')


obj = TrappedResonance(
    eq,
    num_rho=10,
    num_eta=8,
    KE_frac=1,
    num_transit=5,
    num_quad=32,
    num_pitch=16,
    num_well=None,
    pitch_invs=None,
    N=0,
    M=1,
    p_max=10,
    q_max=10,
    res_range_min=-4,
    res_range_max=4,
    DEBUG=False,
    f_q_conservative=False,
    weight_method="bump",
    Delta_Omega=None,
    fill_value=11,
    wd_blur=1.25,
    stab_sacrifice=True,
)
obj.build()
value = obj.compute(eq.params_dict)
print(value)

Precomputing transforms
[  314323.61760635  5244039.62105883  9984002.86053015 12436214.05326557
 10506659.74928156  7426894.55888129  9327658.90897236 13600643.69629207
 24572907.3565026  40083478.0878589 ]


In [3]:
# Example optimization

import sys
import os
import math

sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../../../"))
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.size"] = 20

import desc.io
from desc.grid import LinearGrid, ConcentricGrid
from desc.objectives import (
    ObjectiveFunction,
    FixBoundaryR,
    FixBoundaryZ,
    FixPressure,
    FixIota,
    FixPsi,
    ForceBalance,
    QuasisymmetryTripleProduct,
    GammaC,
    TrappedResonance
)
from desc.optimize import Optimizer
from desc.examples import get



# load initial equilibrium
# eq_init = desc.io.load("desc_eq_beta2.5_QA.h5") # QA
eq_init = get('HELIOTRON')
eq_init.change_resolution(L=4,M=4,N=4) # change resolution to make optimization faster

# Specify equilibrium nfp and helicity
N = 0 # QA
nfp = eq_init.NFP

# optimizer = Optimizer("proximal-scipy-bfgs")
optimizer = Optimizer("proximal-lsq-exact")

# indices of boundary modes we want to optimize
idx_Rcc = eq_init.surface.R_basis.get_idx(M=1, N=2)
idx_Rss = eq_init.surface.R_basis.get_idx(M=-1, N=-2)
idx_Zsc = eq_init.surface.Z_basis.get_idx(M=-1, N=2)
idx_Zcs = eq_init.surface.Z_basis.get_idx(M=1, N=-2)
print("surface.R_basis.modes is an array of [l,m,n] of the surface modes:")
print(eq_init.surface.R_basis.modes[0:10])

# boundary modes to constrain
R_modes = np.delete(eq_init.surface.R_basis.modes, [idx_Rcc, idx_Rss], axis=0)
Z_modes = np.delete(eq_init.surface.Z_basis.modes, [idx_Zsc, idx_Zcs], axis=0)

eq_qs_T = eq_init.copy()  # make a copy of the original equilibria

constraints = (
    ForceBalance(eq=eq_qs_T),  # enforce JxB-grad(p)=0 during optimization
    FixBoundaryR(eq=eq_qs_T, modes=R_modes),  # fix specified R boundary modes
    FixBoundaryZ(eq=eq_qs_T, modes=Z_modes),  # fix specified Z boundary modes
    FixPressure(eq=eq_qs_T),  # fix pressure profile
    FixIota(eq=eq_qs_T),  # fix rotational transform profile
    FixPsi(eq=eq_qs_T),  # fix total toroidal magnetic flux
)

# Create grid for QSTripleProduct objective 
grid_vol = ConcentricGrid(
    L=eq_init.L_grid,
    M=eq_init.M_grid,
    N=eq_init.N_grid,
    NFP=eq_init.NFP,
    sym=eq_init.sym,
)

# Objective for resonance
objective_fT = ObjectiveFunction(
    (
        QuasisymmetryTripleProduct(eq=eq_qs_T, grid=grid_vol),
        TrappedResonance(
            eq,
            num_rho=10,
            num_eta=8,
            KE_frac=1,
            num_transit=5,
            num_quad=32,
            num_pitch=16,
            num_well=None,
            pitch_invs=None,
            N=0,
            M=1,
            p_max=10,
            q_max=10,
            res_range_min=-4,
            res_range_max=4,
            DEBUG=False,
            f_q_conservative=False,
            weight_method="bump",
            Delta_Omega=None,
            fill_value=11,
            wd_blur=1.25,
            stab_sacrifice=True,
        )
    ), 
)

eq_qs_T, result_T = eq_qs_T.optimize(
    objective=objective_fT,
    constraints=constraints,
    optimizer=optimizer,
    ftol=5e-2,  # stopping tolerance on the function value
    xtol=1e-6,  # stopping tolerance on the step size
    gtol=1e-6,  # stopping tolerance on the gradient
    maxiter=10,  # maximum number of iterations
    options={
        "perturb_options": {"order": 2, "verbose": 0},  # use 2nd-order perturbations
        "solve_options": {
            "ftol": 5e-3,
            "xtol": 1e-6,
            "gtol": 1e-6,
            "verbose": 0,
        },  # for equilibrium subproblem
    },
    copy=False,  # copy=False we will overwrite the eq_qs_T object with the optimized result
    verbose=3,
)

surface.R_basis.modes is an array of [l,m,n] of the surface modes:
[[ 0 -4 -4]
 [ 0 -3 -4]
 [ 0 -2 -4]
 [ 0 -1 -4]
 [ 0 -4 -3]
 [ 0 -3 -3]
 [ 0 -2 -3]
 [ 0 -1 -3]
 [ 0 -4 -2]
 [ 0 -3 -2]]
Building objective: QS triple product
Precomputing transforms
Timer: Precomputing transforms = 1.01 sec
Building objective: TrappedResonance
Precomputing transforms
Timer: Precomputing transforms = 212 ms
Timer: Objective build = 3.32 sec
Building objective: force
Precomputing transforms
Timer: Precomputing transforms = 488 ms
Timer: Objective build = 559 ms
Timer: Objective build = 3.87 ms
Timer: Eq Update LinearConstraintProjection build = 5.22 sec
Timer: Proximal projection build = 12.0 sec
Building objective: lcfs R
Building objective: lcfs Z
Building objective: fixed pressure
Building objective: fixed iota
Building objective: fixed Psi
Timer: Objective build = 1.02 sec


AssertionError: 